<a href="https://colab.research.google.com/github/Dharshini11042004/Adaptive-Threat-Detection-in-Smart-Healthcare-Infrastructure-with-Agentic-AI-using-RL/blob/main/Agent1_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AGENT 1 : SMART HEALTHCARE NETWORK IDS
# Autoencoder + Reinforcement Learning (DQN)
!pip install gymnasium stable-baselines3 torch pandas numpy scikit-learn -q
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN
from sklearn.preprocessing import StandardScaler
import random
import os
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# EXTRACT DATASET
zip_path = "preprocessed_Datasets.zip"
with zipfile.ZipFile(zip_path,'r') as z:
    z.extractall()
train_path = "preprocessed_Datasets/Network/Merged01_train.csv"
test_path  = "preprocessed_Datasets/Network/Merged01_test.csv"
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
# PREPROCESSING
X_train = train_df.drop(columns=["Label"])
X_test  = test_df.drop(columns=["Label"])
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print("Dataset extracted")
print("Feature count:", X_train.shape[1])
# AUTOENCODER
class AutoEncoder(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim,32),
            nn.ReLU(),
            nn.Linear(32,16),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(16,32),
            nn.ReLU(),
            nn.Linear(32,input_dim)
        )
    def forward(self,x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out
device = "cuda" if torch.cuda.is_available() else "cpu"
ae = AutoEncoder(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(ae.parameters(),lr=0.001)
criterion = nn.MSELoss()
X_train_tensor = torch.tensor(X_train,dtype=torch.float32).to(device)
for epoch in range(15):
    optimizer.zero_grad()
    recon = ae(X_train_tensor)
    loss = criterion(recon,X_train_tensor)
    loss.backward()
    optimizer.step()
    if epoch%5==0:
        print("AE epoch",epoch,"loss",loss.item())
print("Autoencoder training completed")
# ANOMALY SCORE
def anomaly_score(x):
    x = torch.tensor(x,dtype=torch.float32).to(device)
    with torch.no_grad():
        recon = ae(x)
    err = torch.mean((x-recon)**2,dim=1)
    return err.cpu().numpy()
# THRESHOLDS
normal_th = 0.04
attack_th = 0.10
# RL ENVIRONMENT
class NetworkEnv(gym.Env):
    def __init__(self,data):
        super(NetworkEnv,self).__init__()
        self.data = data
        self.idx = 0
        self.observation_space = spaces.Box(
            low=-10,
            high=10,
            shape=(data.shape[1],),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        self.idx = 0
        return self.data[self.idx],{}
    def step(self,action):
        state = self.data[self.idx]
        score = anomaly_score(state.reshape(1,-1))[0]
        if score < normal_th:
            true_type = 0
        elif score < attack_th:
            true_type = 1
        else:
            true_type = 2
        reward = 5 if action == true_type else -2
        self.idx += 1
        done = self.idx >= len(self.data)-1
        next_state = self.data[self.idx] if not done else state
        return next_state,reward,done,False,{}
# TRAIN RL AGENT
env = NetworkEnv(X_train)
env.reset(seed=SEED)
model = DQN(
    "MlpPolicy",
    env,
    learning_rate=0.0005,
    buffer_size=20000,
    learning_starts=1000,
    batch_size=128,
    gamma=0.99,
    seed=SEED,
    verbose=0
)
model.learn(total_timesteps=20000)
print("RL Training Completed")
# OUTPUT GENERATION (TRAIN + TEST)
def generate_outputs(data, dataset_name):
    env = NetworkEnv(data)
    env.reset(seed=SEED)
    state,_ = env.reset()
    done = False
    outputs = []
    idx = 0
    while not done:
        action,_ = model.predict(state)
        # Compute anomaly score
        score = anomaly_score(state.reshape(1,-1))[0]
        # Risk score (scaled)
        risk_score = float(score * 100)
        # Behaviour mapping
        if action == 0:
            behaviour = "Normal"
            decision = "Allow Traffic"
        elif action == 1:
            behaviour = "Suspicious"
            decision = "Inspect Traffic"
        else:
            behaviour = "Malicious"
            decision = "Block Traffic"
        outputs.append({
            "Dataset": dataset_name,
            "Index": idx,
            "Anomaly Score": float(score),
            "Risk Score": risk_score,
            "Behaviour": behaviour,
            "Action": int(action),
            "Decision": decision
        })
        state,_,done,_,_ = env.step(action)
        idx += 1
    return outputs
# Generate outputs
train_outputs = generate_outputs(X_train, "Train")
test_outputs  = generate_outputs(X_test, "Test")
# Combine
all_outputs = train_outputs + test_outputs
# Save to CSV
output_df = pd.DataFrame(all_outputs)
output_path = "/content/drive/MyDrive/MultiAgentModels/agent_1_outputs.csv"
output_df.to_csv(output_path, index=False)
print("\nSaved → agent_1_outputs.csv")
# EXISTING TEST EVALUATION (UNCHANGED)
env_test = NetworkEnv(X_test)
env_test.reset(seed=SEED)
state,_ = env_test.reset()
done=False
actions = []
rewards = []
while not done:
    action,_ = model.predict(state)
    state,reward,done,_,_ = env_test.step(action)
    actions.append(action)
    rewards.append(reward)
actions = np.array(actions)
allow  = np.sum(actions==0)
inspect = np.sum(actions==1)
block = np.sum(actions==2)
print("\nAction Distribution")
print("ALLOW :",allow)
print("INSPECT :",inspect)
print("BLOCK :",block)
print("\nAverage Reward:",np.mean(rewards))
# ACCURACY
labels = test_df["Label"].values[:len(actions)]
correct = 0
for pred,label in zip(actions,labels):
    if label == "BENIGN":
        if pred == 0:
            correct += 1
    else:
        if pred in [1,2]:
            correct += 1
accuracy = (correct / len(actions)) * 100
print("\nAgent Accuracy:", round(accuracy,2), "%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 5.0 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Dataset extracted
Feature count: 12


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


AE epoch 0 loss 1.0139702558517456


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


AE epoch 5 loss 0.9988625645637512
AE epoch 10 loss 0.983026921749115
Autoencoder training completed
RL Training Completed

Saved → agent_1_outputs.csv

Action Distribution
ALLOW : 16525
INSPECT : 133624
BLOCK : 68599

Average Reward: 4.687453142428731

Agent Accuracy: 90.26 %
